# Day 1: Exploratory Data Analysis (EDA) - LinkedIn Job Postings Dataset

## Overview
This notebook performs comprehensive Exploratory Data Analysis (EDA) on the LinkedIn Job Postings dataset for the **Career Intelligence Engine**.

### 1. Environment Setup & Data Loading

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

RAW_DATA_DIR = os.path.join("..", "Data", "Raw")

print("Loading raw dataset files...")
postings_df = pd.read_csv(os.path.join(RAW_DATA_DIR, "postings.csv"))
companies_df = pd.read_csv(os.path.join(RAW_DATA_DIR, "companies", "companies.csv"))
company_industries_df = pd.read_csv(os.path.join(RAW_DATA_DIR, "companies", "company_industries.csv"))
company_specialities_df = pd.read_csv(os.path.join(RAW_DATA_DIR, "companies", "company_specialities.csv"))
employee_counts_df = pd.read_csv(os.path.join(RAW_DATA_DIR, "companies", "employee_counts.csv"))

benefits_df = pd.read_csv(os.path.join(RAW_DATA_DIR, "jobs", "benefits.csv"))
job_industries_df = pd.read_csv(os.path.join(RAW_DATA_DIR, "jobs", "job_industries.csv"))
job_skills_df = pd.read_csv(os.path.join(RAW_DATA_DIR, "jobs", "job_skills.csv"))
salaries_df = pd.read_csv(os.path.join(RAW_DATA_DIR, "jobs", "salaries.csv"))

industries_df = pd.read_csv(os.path.join(RAW_DATA_DIR, "mappings", "industries.csv"))
skills_df = pd.read_csv(os.path.join(RAW_DATA_DIR, "mappings", "skills.csv"))

print("All 11 dataset files loaded successfully.")

### 2. Dataset Shapes & Overview

In [ ]:
data_summary = [
    {"File": "postings.csv", "Rows": len(postings_df), "Columns": len(postings_df.columns)},
    {"File": "companies.csv", "Rows": len(companies_df), "Columns": len(companies_df.columns)},
    {"File": "company_industries.csv", "Rows": len(company_industries_df), "Columns": len(company_industries_df.columns)},
    {"File": "company_specialities.csv", "Rows": len(company_specialities_df), "Columns": len(company_specialities_df.columns)},
    {"File": "employee_counts.csv", "Rows": len(employee_counts_df), "Columns": len(employee_counts_df.columns)},
    {"File": "benefits.csv", "Rows": len(benefits_df), "Columns": len(benefits_df.columns)},
    {"File": "job_industries.csv", "Rows": len(job_industries_df), "Columns": len(job_industries_df.columns)},
    {"File": "job_skills.csv", "Rows": len(job_skills_df), "Columns": len(job_skills_df.columns)},
    {"File": "salaries.csv", "Rows": len(salaries_df), "Columns": len(salaries_df.columns)},
    {"File": "industries.csv", "Rows": len(industries_df), "Columns": len(industries_df.columns)},
    {"File": "skills.csv", "Rows": len(skills_df), "Columns": len(skills_df.columns)},
]

summary_df = pd.DataFrame(data_summary)
print(summary_df.to_string(index=False))

### 3. Inspection of `postings.csv` (Core Table)

In [ ]:
print("Postings Schema & Data Types:")
print(postings_df.dtypes)
print("\nHead of postings:")
postings_df.head(3)

### 4. Missing Values & Data Quality Inspection

In [ ]:
missing = postings_df.isnull().sum()
missing_pct = (missing / len(postings_df)) * 100
missing_df = pd.DataFrame({"Missing_Count": missing, "Missing_Percentage": missing_pct})
print(missing_df.sort_values(by="Missing_Percentage", ascending=False))

### 5. Identification of CS / Tech Job Roles

In [ ]:
cs_keywords = ["Software", "Developer", "Data", "Engineer", "Computer", "Backend", "Frontend", "Full Stack", "Machine Learning", "AI", "System", "DevOps", "Cloud", "Security", "Database"]
cs_pattern = "|".join(cs_keywords)
cs_jobs = postings_df[postings_df["title"].str.contains(cs_pattern, case=False, na=False)].copy()

print(f"Total Job Postings: {len(postings_df)}")
print(f"CS / Tech Job Postings: {len(cs_jobs)} ({len(cs_jobs)/len(postings_df)*100:.2f}%)")

print("\nTop 15 CS Job Titles:")
print(cs_jobs["title"].value_counts().head(15))

### 6. Experience Levels and Work Types in CS Jobs

In [ ]:
print("CS Job Experience Levels:")
print(cs_jobs["formatted_experience_level"].value_counts(dropna=False))

print("\nCS Job Work Types:")
print(cs_jobs["formatted_work_type"].value_counts(dropna=False))

### 7. High-Level Skill Categories Analysis

In [ ]:
cs_job_ids = set(cs_jobs["job_id"])
cs_skills = job_skills_df[job_skills_df["job_id"].isin(cs_job_ids)].merge(skills_df, on="skill_abr", how="left")

print("Top Skill Categories in CS / Tech Jobs:")
print(cs_skills["skill_name"].value_counts().head(10))

### 8. Salary Distribution Inspection

In [ ]:
salary_data = cs_jobs.dropna(subset=["normalized_salary"])
print(f"CS Postings with Normalized Salary: {len(salary_data)} out of {len(cs_jobs)}")
print(salary_data["normalized_salary"].describe())

### 9. Key Findings & Data Quality Summary
- **Postings Count**: 123,849 total job postings.
- **CS Target Subset**: 26,094 CS/Tech job postings identified via job title pattern matching.
- **Skills Gap**: `skills_desc` is 98% missing in `postings.csv`, and `jobs/job_skills.csv` only maps high-level LinkedIn industry codes (e.g. `IT`, `ENG`). Detailed technical skills (e.g., Python, React, SQL) must be extracted from the unstructured `description` column via NLP/regex extraction during Day 2 Feature Engineering.
- **Salary Data**: ~76% missing salary data. Where available, `normalized_salary` offers clean numerical target data.
- **Experience Level**: 23.7% missing values; must be imputed or inferred from job descriptions.